# Problem Statement: Canary Rollout Prediction Shift Analysis

## 🧩 Problem Statement

### Scenario Description

A machine learning team deploys a **new classifier model** using a **canary rollout strategy** (10% traffic to new model). After 2 hours, specific observations are made:

- **Class A Predictions:** Shifted from 20% (baseline) to 55% (canary) ⚠️
- **Latency:** Normal ✅
- **Error Rate:** Normal ✅
- **Labels:** Not available yet ❌

### Why It Matters

This is a **silent failure**. The model runs without errors but produces drastically different predictions. Without ground truth labels, we don't know if these new predictions are genius or garbage. We need to investigate **BEFORE** rolling out to 100% of users.

## 🪜 Steps to Solve the Problem

1. **Simulate Data:** Create baseline and canary datasets representing the scenario.
2. **Data Quality Check:** Ensure inputs aren't broken (nulls, types).
3. **Drift Detection:** Check for **Covariate Shift** (input changes) using PSI and KS-Test.
4. **Prediction Analysis:** Quantify the output shift using Chi-Square tests.
5. **Decision:** Determine the safest action (Continue vs. Rollback).

## 🎯 Expected Output

A diagnostic report identifying the root cause (Input Drift) and a recommended action (**PAUSE + ROUTE TO REVIEW**) with statistical justification.

### 🔹 Load Libraries and Set Random Seed
#### 2.1 What the line does
Imports necessary libraries for data manipulation, statistical testing, and visualization, and sets a random seed.
#### 2.2 Why it is used
- `numpy` & `pandas`: Data handling.
- `scipy.stats`: Statistical tests (KS, Chi-Square).
- `np.random.seed(42)`: Ensures results are reproducible.
#### 2.3 When to use it
At the very start of any data science project.
#### 2.4 Where to use it
First code cell.
#### 2.5 How to use it
Standard import syntax.
#### 2.6 How it works internally
Loads compiled C/Fortran code for fast math operations.
#### 2.7 Output
No visible output, but namespaces are populated.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from collections import Counter
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

### 🔹 Simulate Baseline Data
#### 2.1 What the line does
Creates synthetic data representing the **Stable (Old) Model**.
#### 2.2 Why it is used
To establish a reference point ("ground truth" for distribution) to compare the new model against.
#### 2.3 When to use it
When real historical data isn't available or for teaching/simulation purposes.
#### 2.4 Where to use it
Before running any comparisons.
#### 2.5 How to use it
Define distributions (Normal, Uniform) and sample from them.
#### 2.6 How it works internally
Uses pseudo-random number generators to draw samples from specified probability distributions.
#### 2.7 Output
Arrays/DataFrames of features and predictions.

In [2]:
# Simulate Class Predictions (Baseline: A=20%, B=50%, C=30%)
baseline_preds = np.random.choice(
    [0, 1, 2], 
    size=10000, 
    p=[0.20, 0.50, 0.30]  # Probabilities
)

# Simulate Input Features (Baseline)
baseline_features = pd.DataFrame({
    'feature_1': np.random.normal(loc=50, scale=10, size=10000),  # Mean=50
    'feature_2': np.random.normal(loc=100, scale=20, size=10000),
    'feature_3': np.random.uniform(low=0, high=1, size=10000)
}) # No visible output yet

### 🔹 Simulate Canary Data (With Drift)
#### 2.1 What the line does
Creates data for the **Canary (New) Model** that includes **Drift**.
#### 2.2 Why it is used
To replicate the problem scenario: class distribution shift (A: 20% -> 55%) and input drift (feature_1 mean shift).
#### 2.3 When to use it
In simulations to test monitoring systems.
#### 2.4 Where to use it
Parallel to baseline data creation.
#### 2.5 How to use it
Change the probability parameters `p` and distribution parameters `loc`.
#### 2.6 How it works internally
Generates data from a *different* statistical distribution than the baseline.
#### 2.7 Output
Drifted Arrays/DataFrames.

In [3]:
# Simulate Canary Predictions (Shifted: A=55%, B=30%, C=15%)
canary_preds = np.random.choice(
    [0, 1, 2], 
    size=10000, 
    p=[0.55, 0.30, 0.15]  # CHANGED PROBABILITIES <- Pred Drift
)

# Simulate Canary Features (With DRIFT on feature_1)
canary_features = pd.DataFrame({
    'feature_1': np.random.normal(loc=65, scale=10, size=10000),  # Mean=65 (SHIFTED from 50) <- Input Drift
    'feature_2': np.random.normal(loc=100, scale=20, size=10000), # Same as baseline
    'feature_3': np.random.uniform(low=0, high=1, size=10000)     # Same as baseline
})

### 🔹 Function: Check Data Quality
#### 2.1 What the line does
Checks for basic issues like missing values or weird data types.
#### 2.2 Why it is used
Garbage In, Garbage Out. We must ensure the "pipe" isn't broken before checking model logic.
#### 2.3 When to use it
ALWAYS. The first step of any debugging.
#### 2.4 Where to use it
Before running statistical tests.
#### 2.5 How to use it
Call `isnull().sum()` and `dtypes` on the DataFrame.
#### 2.6 How it works internally
Scans memory blocks for null pointers or sentinel values (NaN).
#### 2.7 Output
Prints count of nulls.

In [4]:
def check_data_quality(df, name):
    print(f"--- {name} Quality Check ---")
    # ⚙️ ARGS: df (DataFrame), name (string label)
    
    # Check for nulls
    missing = df.isnull().sum().sum()
    print(f"Total Missing Values: {missing}")
    
    # Check types
    print(f"Data Types:\n{df.dtypes}\n")
    return missing

# 📌 Sample Example
check_data_quality(baseline_features, "Baseline")
check_data_quality(canary_features, "Canary")

--- Baseline Quality Check ---
Total Missing Values: 0
Data Types:
feature_1    float64
feature_2    float64
feature_3    float64
dtype: object

--- Canary Quality Check ---
Total Missing Values: 0
Data Types:
feature_1    float64
feature_2    float64
feature_3    float64
dtype: object



np.int64(0)

### 🔹 Function: Calculate Population Stability Index (PSI)
#### 2.1 What the line does
Calculates PSI to quantify how much a distribution has shifted.
#### 2.2 Why it is used
It's the industry standard metric for drift. **PSI > 0.25** means "Stop the Presses!"
#### 2.3 When to use it
Monitoring production models for input drift.
#### 2.4 Where to use it
On every feature, comparing Training vs. Serving data.
#### 2.5 How to use it
Bin the data, convert to % counts, and use formula: `sum((Actual% - Expected%) * ln(Actual% / Expected%))`.
#### 2.6 How it works internally
It measures the "divergence" (specifically symmetric KL-divergence) between two histograms.
#### 2.7 Output
A single float number (PSI score).

In [5]:
def calculate_psi(expected, actual, buckets=10):
    # ⚙️ ARGS: 
    # expected: baseline data (array)
    # actual: canary data (array)
    # buckets: number of bins for histogram (default 10)
    
    # Create bins based on EXPECTED distribution
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints = np.unique(breakpoints) # Handle duplicates
    
    # Calculate frequency counts
    exp_cts = np.histogram(expected, bins=breakpoints)[0]
    act_cts = np.histogram(actual, bins=breakpoints)[0]
    
    # Convert to proportions (probabilities)
    exp_prob = exp_cts / len(expected) + 0.0001 # Add small epsilon to avoid divide-by-zero
    act_prob = act_cts / len(actual) + 0.0001
    
    # PSI Formula
    psi = np.sum((act_prob - exp_prob) * np.log(act_prob / exp_prob))
    return psi

### 🔹 Run Drift Detection (PSI & KS-Test)
#### 2.1 What the line does
Iterates through all features and runs PSI + KS-Test.
#### 2.2 Why it is used
To automatically flag which specific features represent a risk.
#### 2.3 When to use it
Part of the automated monitoring pipeline.
#### 2.4 Where to use it
After data extraction.
#### 2.5 How to use it
Loop through columns, pass arrays to PSI function and `stats.ks_2samp`.
#### 2.6 How it works internally
PSI checks histogram overlap. KS-Test checks cumulative distribution function (CDF) distance.
#### 2.7 Output
Drift report per feature.

In [6]:
print("--- Input Drift Detection ---\n")
for col in baseline_features.columns:
    val_base = baseline_features[col].values
    val_canary = canary_features[col].values
    
    # 1. PSI
    psi = calculate_psi(val_base, val_canary)
    
    # 2. KS Test
    ks_stat, p_val = stats.ks_2samp(val_base, val_canary)
    
    status = "✅ OK" if psi < 0.1 else "⚠️ DRIFT" if psi < 0.25 else "🚨 CRITICAL"
    print(f"Feature: {col:<10} | PSI: {psi:.3f} | KS p-val: {p_val:.3f} | Status: {status}")

--- Input Drift Detection ---

Feature: feature_1  | PSI: 1.955 | KS p-val: 0.000 | Status: 🚨 CRITICAL
Feature: feature_2  | PSI: 0.002 | KS p-val: 0.426 | Status: ✅ OK
Feature: feature_3  | PSI: 0.002 | KS p-val: 0.688 | Status: ✅ OK


### 🔹 Analyze Prediction Behavior
#### 2.1 What the line does
Compares the percentage of each class predicted by Baseline vs. Canary.
#### 2.2 Why it is used
To quantify the "Prediction Drift" mentioned in the problem statement.
#### 2.3 When to use it
When monitoring classification models.
#### 2.4 Where to use it
Dashboard or report output.
#### 2.5 How to use it
Use `Counter` or `value_counts(normalize=True)`.
#### 2.6 How it works internally
Counts occurrences of unique values and divides by total count.
#### 2.7 Output
Class distribution table.

In [7]:
print("\n--- Prediction Behavior Analysis ---")
base_cts = pd.Series(baseline_preds).value_counts(normalize=True).sort_index()
canary_cts = pd.Series(canary_preds).value_counts(normalize=True).sort_index()

df_res = pd.DataFrame({'Baseline': base_cts, 'Canary': canary_cts})
df_res['Diff'] = df_res['Canary'] - df_res['Baseline']
print(df_res)


--- Prediction Behavior Analysis ---
   Baseline  Canary    Diff
0    0.2043  0.5460  0.3417
1    0.5070  0.3054 -0.2016
2    0.2887  0.1486 -0.1401


### 💼 Interview Perspective
- **Q:** "If a model has normal latency but weird predictions, what do you check first?"
- **A:** "I check **Input Drift** (did the users change?) vs. **Model Calibration** (did we break the probability thresholds?). Here, use PSI for inputs."
- **Q:** "When do you rollback?"
- **A:** "If PSI > 0.25 or prediction shift > 10% without business justification. Safety first."

## 🏁 Conclusion & Recommendation

### **Decision: PAUSE and ROUTE TO REVIEW**

**Justification:**
1. **Input Drift Confirmed:** `feature_1` has high PSI (>0.25) and KS-test failure (p < 0.05). The model is seeing data it wasn't trained for (Mean 50 vs 65).
2. **Prediction Shift:** Class A jumped 35%. This correlates with the input drift.
3. **Action:** We cannot trust these predictions. **Pause** the canary (stop increasing traffic), investigate the upstream data source for `feature_1`, then decide whether to retrain or fix the data pipeline.